In [ ]:
import gmsh
import sys
import os
from dotenv import load_dotenv 

In [ ]:
load_dotenv(dotenv_path='/home/asus_juan/Documents/GitHub/Despliegue-modelo-ANN-Turbinas-Hidraulicas/.env')
CAD = os.getenv('CAD')

In [ ]:
def create_mesh_impulsor(ruta_cad, ruta_salida):

    # Para usar gmsh es necesario inicializarlo
    gmsh.initialize()

    # Creamos una configuracion que nos arroja el progreso en la termina
    gmsh.option.set_number("General.Terminal", 1)
    gmsh.model.add("Fluido_Turbina")

    # Creamos el impulsor como geometria
    # importShapes devuelve una lista de Tuplas 
    impulsor = gmsh.model.occ.importShapes(ruta_cad)

    # Creamos el volumen del agua una gran ventaja de las turbinas esque podemos colocar el agua como un cilindro
    # IMPORTANTE: el que vaya a ajustar estas coordenadas debe garantizar que el cilindro de agua envuelva completamente el impulsor 
    tag_cilinder = gmsh.model.occ.addCylinder() # Necesario colocar los datos del cad (Dejarselo a alguien que quiera, me da pereza abrir Ansys, tengo Linux jejeje)
    vol_water = [(3, tag_cilinder)] 

    # Al cilindro de agua le quitamos el epacio que ocupa el impulsor
    dominio_flujo, transofrmation_matrix = gmsh.model.occ.cut(vol_water, impulsor)

    # Ya que gmsh es un modelo matematico debe sinconizarse 
    gmsh.model.occ.synchronize()

    # Estos son los valores que pueden tomar las mallas
    gmsh.option.setNumber("Mesh.MeshSizeMin", 2.0)
    gmsh.option.setNumber("Mesh.MeshSizeMax", 8.0)
    
    # Le indicamos a Gmsh que genere una malla tridimensional
    gmsh.model.mesh.generate(3)

    # Guardamos el modelo .mesh
    try:
        gmsh.write(ruta_salida)
        print('Guardado Correctamente')
    except Exception as e:
        print(f'Hubo error en guardar verifica: {e}')
    
    if '-nopopup' not in sys.argv:
        gmsh.fltk.run()

    gmsh.finalize()

In [ ]:
if __name__ == "__main__":
    create_mesh_impulsor(CAD, '/home/asus_juan/Documents/GitHub/Despliegue-modelo-ANN-Turbinas-Hidraulicas/CADS_Models/MESH')